In [1]:
# PS 1:
# Compare the performance of a model trained from scratch with a pretrained model on the same dataset.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# subset for CPU
x_train = x_train[:2000]
y_train = y_train[:2000]

x_test = x_test[:500]
y_test = y_test[:500]

x_train = x_train / 255.0
x_test = x_test / 255.0

# CNN from scratch
cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.fit(x_train, y_train, epochs=2)

cnn_loss, cnn_acc = cnn_model.evaluate(x_test, y_test)

print("CNN Accuracy:", cnn_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.1580 - loss: 2.2412
Epoch 2/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2775 - loss: 1.9899
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.3080 - loss: 1.9016 
CNN Accuracy: 0.30799999833106995


In [2]:
# PS 2:
# Analyse differences in training time and accuracy between the two approaches.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import time

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train[:2000] / 255.0
y_train = y_train[:2000]

x_test = x_test[:500] / 255.0
y_test = y_test[:500]

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()

model.fit(x_train, y_train, epochs=2)

end = time.time()

loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)
print("Training Time:", end - start, "seconds")

Epoch 1/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.2180 - loss: 2.1072
Epoch 2/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3455 - loss: 1.8112
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3180 - loss: 1.8424
Accuracy: 0.3179999887943268
Training Time: 3.2766106128692627 seconds


In [3]:
# PS 3:
# Evaluate how the dataset size impacts both methods.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# small dataset
x_train = x_train[:500]
y_train = y_train[:500]

x_test = x_test[:200]
y_test = y_test[:200]

x_train = x_train / 255.0
x_test = x_test / 255.0

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2)

loss, acc = model.evaluate(x_test, y_test)

print("Accuracy with small dataset:", acc)

Epoch 1/2
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.1060 - loss: 2.4173
Epoch 2/2
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1100 - loss: 2.2722 
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.1050 - loss: 2.2531
Accuracy with small dataset: 0.10499999672174454


In [4]:
# PS 4:
# Identify scenarios where transfer learning is more beneficial.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# very small dataset
x_train = x_train[:1000]
y_train = y_train[:1000]

x_test = x_test[:300]
y_test = y_test[:300]

x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2)

loss, acc = model.evaluate(x_test, y_test)

print("Transfer Learning Accuracy:", acc)
print("Transfer learning is useful for small datasets.")

Epoch 1/2
32/32 ━━━━━━━━━━━━━━━━━━━━ 28s 353ms/step - accuracy: 0.3160 - loss: 2.0910
Epoch 2/2
32/32 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.6190 - loss: 1.0858
10/10 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.6133 - loss: 1.1116
Transfer Learning Accuracy: 0.6133333444595337
Transfer learning is useful for small datasets.
